## Neuron pruning experiment (gradient × weight saliency)

Following the paper: compute gradients of the **pretraining task loss** (count_b) w.r.t. model weights,
rank MLP neurons by `grad · weight` saliency, then zero out the top-k most salient neurons.
If finetuning learned a "wrapper", pruning a small number of neurons should recover count_b performance.

In [45]:
import sys, os
sys.path.insert(0, '/workspace/PCFG')
import torch
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from mingpt import GPT, GPTConfig
from pcfg_gen import CharTokenizer, PCFGGenerator, PCFGDataset, build_pools, format_example, collate_fn
from config import CFG
from config_utils import build_task_registry, set_seed
from train_help import _evaluate_loader
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ---- experiment grid ----
CORRS = CFG['experiment']['correlation_values']
CONCS = CFG['experiment']['concentration_values']

VAL_SPLITS = [
    'count_a_corr', 'count_a_uncorr',
    'count_b_corr', 'count_b_uncorr',
    'all_other_corr', 'all_other_uncorr',
]
SPLIT_LABELS = {
    'count_a_corr':     'count_a (corr)',
    'count_a_uncorr':   'count_a (uncorr)',
    'count_b_corr':     'count_b (corr)',
    'count_b_uncorr':   'count_b (uncorr)',
    'all_other_corr':   'other tasks (corr)',
    'all_other_uncorr': 'other tasks (uncorr)',
}

# ---- tokenizer & model config ----
tokenizer = CharTokenizer()
mcfg = CFG['model']
gpt_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=mcfg['block_size'],
    n_layer=mcfg['n_layer'],
    n_head=mcfg['n_head'],
    n_embd=mcfg['n_embd'],
    embd_pdrop=0.0, resid_pdrop=0.0, attn_pdrop=0.0,
)

# ---- build eval loaders ----
set_seed(42)
pcfg = PCFGGenerator()
task_registry = build_task_registry(CFG['task_definitions'])

print("Building string pools...")
pools = build_pools(pcfg_gen=pcfg, n_correlated=5000, n_uncorrelated=5000,
                    chunk_size=250, verbose=True)

N_EVAL = 200
max_len = CFG['tokenizer']['max_length']

def make_ds(pool, tasks, n):
    import random
    examples = []
    for _ in range(n):
        s = random.choice(pool)
        t = random.choice(tasks)
        td, ans = task_registry.apply_task(t, s)
        examples.append(format_example(s, td, ans))
    return PCFGDataset(examples, tokenizer, max_length=max_len, mask_answer_only=True)

other_tasks = [t for t in CFG['task_sets']['all'] if t not in ['count_a', 'count_b']]

eval_datasets = {
    'count_a_corr':     make_ds(pools['correlated'],   ['count_a'], N_EVAL),
    'count_a_uncorr':   make_ds(pools['uncorrelated'], ['count_a'], N_EVAL),
    'count_b_corr':     make_ds(pools['correlated'],   ['count_b'], N_EVAL),
    'count_b_uncorr':   make_ds(pools['uncorrelated'], ['count_b'], N_EVAL),
    'all_other_corr':   make_ds(pools['correlated'],   other_tasks, N_EVAL * len(other_tasks)),
    'all_other_uncorr': make_ds(pools['uncorrelated'], other_tasks, N_EVAL * len(other_tasks)),
}
eval_loaders = {
    name: DataLoader(ds, batch_size=64, shuffle=False,
                     collate_fn=lambda b, tok=tokenizer: collate_fn(b, tok))
    for name, ds in eval_datasets.items()
}
print("Eval sets:", {k: len(v.dataset) for k, v in eval_loaders.items()})


Device: cuda
Tokenizer vocabulary size: 267
Building string pools...
Building PCFG pools: 5,000 correlated + 5,000 uncorrelated strings (window=40) …
Pools ready — 5,000 correlated, 5,000 uncorrelated from 27,284 total generations (18.3% acceptance rate).
Eval sets: {'count_a_corr': 200, 'count_a_uncorr': 200, 'count_b_corr': 200, 'count_b_uncorr': 200, 'all_other_corr': 2200, 'all_other_uncorr': 2200}


In [46]:
PRETRAIN_DIR = '/workspace/PCFG'
MODELS_DIR   = '/workspace/PCFG/results/models'

def load_state(path):
    ck = torch.load(path, map_location='cpu')
    return ck['model_state_dict']

def compute_neuron_saliency(model, loaders_with_weights, device, n_layers=6):
    """
    Compute gradient * weight saliency for each MLP neuron.
    loaders_with_weights: list of (loader, weight) tuples.
      The loss is a weighted sum across loaders, matching the training data mix.
    Returns array of shape (n_layers, n_neurons).
    """
    model.train()
    model.zero_grad()

    for loader, w in loaders_with_weights:
        if w == 0:
            continue
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['target_ids'].to(device)
            logits, loss = model(input_ids, target_ids)
            (loss * w).backward()

    n_neurons = 4 * mcfg['n_embd']  # 768
    saliency = np.zeros((n_layers, n_neurons))

    for l in range(n_layers):
        fc_w = model.transformer.h[l].mlp.c_fc.weight
        neuron_fc_sal = (fc_w.grad * fc_w).detach().sum(dim=1).cpu().numpy()

        fc_b = model.transformer.h[l].mlp.c_fc.bias
        neuron_bias_sal = (fc_b.grad * fc_b).detach().cpu().numpy()

        proj_w = model.transformer.h[l].mlp.c_proj.weight
        neuron_proj_sal = (proj_w.grad * proj_w).detach().sum(dim=0).cpu().numpy()

        saliency[l] = neuron_fc_sal + neuron_bias_sal + neuron_proj_sal

    model.zero_grad()
    return saliency

def prune_neurons(state_dict, saliency, k):
    """Zero out top-k neurons by saliency score."""
    state = deepcopy(state_dict)
    n_layers, n_neurons = saliency.shape
    flat = saliency.flatten()
    if k == 0:
        return state
    selected = np.argsort(flat)[::-1][:k]
    for idx in selected:
        l = idx // n_neurons
        j = idx %  n_neurons
        fc_key   = f'transformer.h.{l}.mlp.c_fc.weight'
        proj_key = f'transformer.h.{l}.mlp.c_proj.weight'
        fc_bias  = f'transformer.h.{l}.mlp.c_fc.bias'
        state[fc_key][j, :]   = 0.0
        state[proj_key][:, j] = 0.0
        if fc_bias in state:
            state[fc_bias][j] = 0.0
    return state

def eval_state(state_dict, loaders, device):
    model = GPT(gpt_config).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    results = {}
    for name, loader in loaders.items():
        loss, acc = _evaluate_loader(model, loader, device, metrics_set={'loss', 'answer_acc'})
        results[name] = {'loss': loss, 'acc': acc}
    return results

# ---- Discover available model files ----
import glob as glob_mod
available_models = glob_mod.glob(f'{MODELS_DIR}/finetune_corr_*_conc_*.pth')
print(f'Found {len(available_models)} finetune checkpoints:')
for p in sorted(available_models):
    print(f'  {os.path.basename(p)}')

Found 10 finetune checkpoints:
  finetune_corr_0.95_conc_0.90_seed2.pth
  finetune_corr_0.95_conc_0.92_seed2.pth
  finetune_corr_0.95_conc_0.95_seed2.pth
  finetune_corr_0.95_conc_0.98_seed2.pth
  finetune_corr_0.95_conc_1.00_seed2.pth
  finetune_corr_1.00_conc_0.90_seed2.pth
  finetune_corr_1.00_conc_0.92_seed2.pth
  finetune_corr_1.00_conc_0.95_seed2.pth
  finetune_corr_1.00_conc_0.98_seed2.pth
  finetune_corr_1.00_conc_1.00_seed2.pth


In [39]:

import sys
sys.path.insert(0, '/workspace/PCFG')
from style import apply_paper_style, figsize, save_figure
import matplotlib.lines as mlines

TARGET_CORRS = [1.0, 0.95]
TARGET_CONCS = [0.9, 0.95, 1.0]
K_VALS = [0, 5, 10, 20]

# ---- Run pruning for target subset ----
target_pruning = {}

for corr in TARGET_CORRS:
    for conc in TARGET_CONCS:
        pair_name = f'corr_{corr:.2f}_conc_{conc:.2f}'
        ft_path = None
        for suffix in ['', '_seed2', '_seed124']:
            candidate = f'{MODELS_DIR}/finetune_{pair_name}{suffix}.pth'
            if os.path.exists(candidate):
                ft_path = candidate
                break
        if ft_path is None:
            print(f'Skipping {pair_name} — no checkpoint found')
            continue

        print(f'Pruning {pair_name}')
        ft_state = load_state(ft_path)

        model = GPT(gpt_config).to(device)
        model.load_state_dict(ft_state)
        saliency = compute_neuron_saliency(
            model,
            [(eval_loaders['count_b_corr'], corr), (eval_loaders['count_b_uncorr'], 1 - corr)],
            device,
        )
        del model

        target_pruning[pair_name] = {}
        for k in K_VALS:
            pruned = prune_neurons(ft_state, saliency, k=k)
            res = eval_state(pruned, eval_loaders, device)
            target_pruning[pair_name][k] = res
            print(f'  k={k:4d}  count_a={corr*res["count_a_corr"]["acc"]+(1-corr)*res["count_a_uncorr"]["acc"]:.3f}'
                  f'  count_b={corr*res["count_b_corr"]["acc"]+(1-corr)*res["count_b_uncorr"]["acc"]:.3f}')

print('\nDone!')



Pruning corr_1.00_conc_0.90
number of parameters: 2.82M
number of parameters: 2.82M
  k=   0  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=   5  count_a=0.965  count_b=0.625
number of parameters: 2.82M
  k=  10  count_a=0.935  count_b=0.295
number of parameters: 2.82M
  k=  20  count_a=0.855  count_b=0.165
Pruning corr_1.00_conc_0.95
number of parameters: 2.82M
number of parameters: 2.82M
  k=   0  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=   5  count_a=0.950  count_b=0.030
number of parameters: 2.82M
  k=  10  count_a=0.615  count_b=0.000
number of parameters: 2.82M
  k=  20  count_a=0.425  count_b=0.000
Pruning corr_1.00_conc_1.00
number of parameters: 2.82M
number of parameters: 2.82M
  k=   0  count_a=1.000  count_b=0.000
number of parameters: 2.82M
  k=   5  count_a=0.855  count_b=0.945
number of parameters: 2.82M
  k=  10  count_a=0.115  count_b=0.960
number of parameters: 2.82M
  k=  20  count_a=0.165  count_b=0.370
Pruning corr_0.95_conc_0.90


In [40]:

# ---- Plot ----
apply_paper_style()
plt.rcParams.update({'font.size': 14, 'axes.titlesize': 16,
                     'axes.labelsize': 14, 'xtick.labelsize': 13, 'ytick.labelsize': 13})

CONC_COLORS_PLOT = plt.cm.viridis(np.linspace(0.2, 0.85, len(TARGET_CONCS)))

row_specs = [
    ('count_a', 'Count-A accuracy'),
    ('count_b', 'Count-B accuracy'),
]

fw, fh = figsize('full', nrows=len(row_specs))
fig, axes = plt.subplots(
    len(row_specs), len(TARGET_CORRS),
    figsize=(fw, fh * 0.7),
    sharey='row', sharex='all'
)

for row, (task, ylabel) in enumerate(row_specs):
    for col, corr in enumerate(TARGET_CORRS):
        ax = axes[row, col]
        for conc, color in zip(TARGET_CONCS, CONC_COLORS_PLOT):
            pair_name = f'corr_{corr:.2f}_conc_{conc:.2f}'
            if pair_name not in target_pruning:
                continue
            ks = sorted(target_pruning[pair_name].keys())
            weighted_accs = [
                corr       * target_pruning[pair_name][k][f'{task}_corr']['acc']
                + (1-corr) * target_pruning[pair_name][k][f'{task}_uncorr']['acc']
                for k in ks
            ]
            ax.plot(ks, weighted_accs, 'o-', color=color, linewidth=1.0,
                    markersize=3, label=f'conc = {conc:.2f}')

        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(fr'$\rho$ = {corr:.2f}')
        if row == len(row_specs) - 1:
            ax.set_xlabel('Neurons pruned (k)')
        if col == 0:
            ax.set_ylabel(ylabel)

handles = [
    mlines.Line2D([], [], color=c, marker='o', markersize=3,
                  linewidth=1.0, label=fr'$c = ${conc:.2f}')
    for conc, c in zip(TARGET_CONCS, CONC_COLORS_PLOT)
]
fig.legend(handles=handles, loc='lower center', ncol=len(TARGET_CONCS),
           bbox_to_anchor=(0.5, -0.06), fontsize=12)

save_figure(fig, '/workspace/PCFG/results/plots/pruning_acc_count_ab_high_corr')
plt.show()
print('Saved -> results/plots/pruning_acc_count_ab_high_corr.pdf')


Saved -> results/plots/pruning_acc_count_ab_high_corr.pdf


In [47]:
# ---- Leave-one-out (LOO): compute ----
# For each neuron, hook before c_proj to zero its post-GELU activation and measure
# the actual change in weighted count_b accuracy (no approximation).
# saliency[l,j] = count_b_acc_after_zeroing_j - baseline
# Positive = zeroing helps count_b → genuine suppressor. Select top-k.
#
# LOO_N_BATCHES controls how many batches (of batch_size=64) are used per split
# when measuring each neuron's effect. Higher = less noisy ranking, slower run.
# At low correlation the per-neuron effect is subtle, so a larger eval set helps.

LOO_N_BATCHES = 3  # 3 batches * 64 = 192 examples per corr/uncorr split

def compute_loo_saliency(ft_state, eval_loaders, corr, device, n_layers=6, n_batches=LOO_N_BATCHES):
    model = GPT(gpt_config).to(device)
    model.load_state_dict(ft_state)
    model.eval()
    n_neurons = 4 * mcfg['n_embd']

    def collect(loader, n):
        out = []
        for i, b in enumerate(loader):
            if i >= n:
                break
            out.append((b['input_ids'].to(device), b['target_ids'].to(device)))
        return out

    bc = collect(eval_loaders['count_b_corr'],   n_batches)
    bu = collect(eval_loaders['count_b_uncorr'], n_batches)

    def eval_acc():
        cc = tc = cu = tu = 0
        with torch.no_grad():
            for inp, tgt in bc:
                logits, _ = model(inp, tgt)
                mask = tgt != -100
                cc += (logits.argmax(-1)[mask] == tgt[mask]).sum().item()
                tc += mask.sum().item()
            for inp, tgt in bu:
                logits, _ = model(inp, tgt)
                mask = tgt != -100
                cu += (logits.argmax(-1)[mask] == tgt[mask]).sum().item()
                tu += mask.sum().item()
        return corr * (cc / tc if tc else 0.0) + (1 - corr) * (cu / tu if tu else 0.0)

    baseline = eval_acc()
    saliency = np.zeros((n_layers, n_neurons))

    for l in range(n_layers):
        print(f'  layer {l}', end='', flush=True)
        mlp = model.transformer.h[l].mlp
        for j in range(n_neurons):
            def make_hook(idx):
                def hook(module, inp):
                    x = inp[0].clone()
                    x[..., idx] = 0.0
                    return (x,)
                return hook
            handle = mlp.c_proj.register_forward_pre_hook(make_hook(j))
            saliency[l, j] = eval_acc() - baseline
            handle.remove()
        print(' done')

    return saliency, baseline

pruning_loo = {}

for corr in TARGET_CORRS:
    for conc in TARGET_CONCS:
        pair_name = f'corr_{corr:.2f}_conc_{conc:.2f}'
        ft_path = None
        for suffix in ['', '_seed2', '_seed124']:
            candidate = f'{MODELS_DIR}/finetune_{pair_name}{suffix}.pth'
            if os.path.exists(candidate):
                ft_path = candidate
                break
        if ft_path is None:
            print(f'Skipping {pair_name} — no checkpoint found')
            continue

        print(f'Computing LOO saliency for {pair_name}  (n_batches={LOO_N_BATCHES})')
        ft_state = load_state(ft_path)
        saliency, baseline = compute_loo_saliency(ft_state, eval_loaders, corr, device)
        n_pos = int((saliency > 0).sum())
        print(f'  baseline count_b={baseline:.3f}  positive-LOO neurons={n_pos}/{saliency.size}')

        pruning_loo[pair_name] = {}
        for k in K_VALS:
            pruned = prune_neurons(ft_state, saliency, k=k)
            res = eval_state(pruned, eval_loaders, device)
            pruning_loo[pair_name][k] = res
            print(f'  k={k:4d}  count_a={corr*res["count_a_corr"]["acc"]+(1-corr)*res["count_a_uncorr"]["acc"]:.3f}'
                  f'  count_b={corr*res["count_b_corr"]["acc"]+(1-corr)*res["count_b_uncorr"]["acc"]:.3f}')

print('\nDone!')


Computing LOO saliency for corr_1.00_conc_0.90  (n_batches=3)
number of parameters: 2.82M
  layer 0 done
  layer 1 done
  layer 2 done
  layer 3 done
  layer 4 done
  layer 5 done
  baseline count_b=1.000  positive-LOO neurons=0/4608
number of parameters: 2.82M
  k=   0  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=   5  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=  10  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=  20  count_a=1.000  count_b=1.000
Computing LOO saliency for corr_1.00_conc_0.95  (n_batches=3)
number of parameters: 2.82M
  layer 0 done
  layer 1 done
  layer 2 done
  layer 3 done
  layer 4 done
  layer 5 done
  baseline count_b=1.000  positive-LOO neurons=0/4608
number of parameters: 2.82M
  k=   0  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=   5  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=  10  count_a=1.000  count_b=1.000
number of parameters: 2.82M
  k=  20  count_a=1.000  count_b=1.

In [49]:
# ---- Leave-one-out (LOO): plot ----
apply_paper_style()
plt.rcParams.update({'font.size': 14, 'axes.titlesize': 16,
                     'axes.labelsize': 14, 'xtick.labelsize': 13, 'ytick.labelsize': 13})

fw, fh = figsize('full', nrows=len(row_specs))
fig, axes = plt.subplots(
    len(row_specs), len(TARGET_CORRS),
    figsize=(fw, fh * 0.7),
    sharey='row', sharex='all'
)
# fig.suptitle('Leave-one-out: prune neurons whose ablation most improves count_b', fontsize=12, y=1.02)

for row, (task, ylabel) in enumerate(row_specs):
    for col, corr in enumerate(TARGET_CORRS):
        ax = axes[row, col]
        for conc, color in zip(TARGET_CONCS, CONC_COLORS_PLOT):
            pair_name = f'corr_{corr:.2f}_conc_{conc:.2f}'
            if pair_name not in pruning_loo:
                continue
            ks = sorted(pruning_loo[pair_name].keys())
            weighted_accs = [
                corr       * pruning_loo[pair_name][k][f'{task}_corr']['acc']
                + (1-corr) * pruning_loo[pair_name][k][f'{task}_uncorr']['acc']
                for k in ks
            ]
            ax.plot(ks, weighted_accs, 'o-', color=color, linewidth=1.0,
                    markersize=3)
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(fr'$\rho$ = {corr:.2f}')
        if row == len(row_specs) - 1:
            ax.set_xlabel('Neurons pruned (k)')
        if col == 0:
            ax.set_ylabel(ylabel)

handles = [
    mlines.Line2D([], [], color=c, marker='o', markersize=3,
                  linewidth=1.0, label=f'c = {conc:.2f}')
    for conc, c in zip(TARGET_CONCS, CONC_COLORS_PLOT)
]
fig.legend(handles=handles, loc='lower center', ncol=len(TARGET_CONCS),
           bbox_to_anchor=(0.5, -0.06), fontsize=12)
save_figure(fig, '/workspace/PCFG/results/plots/pruning_loo')
plt.show()
print('Saved -> results/plots/pruning_loo.pdf')


Saved -> results/plots/pruning_loo.pdf
